# CF18 — Validation Audit Notebook

This notebook audits **CF18: Scale Re-Articulation, Effective Action, Hidden-Return Corrections, Running Couplings, Renormalization, and Corrected Symmetry Structure in VDM** using a claim-manifest and strict numerical gates. The source paper is external; this notebook carries only the burdens it attacks.

Primitive driver: **A(-1)** — Universal Axiom of Primitive Bifurcation.

## Audit contract

The notebook walks the paper's closure burdens in order. Each audited unit starts in plain language, states the exact claim being attacked, defines explicit pass criteria, and says what the attack still cannot prove.

Hard rule obeyed here: the notebook does **not** embed raw TeX sections, bibliography blocks, or cloned paper body text. It carries a lean burden manifest plus executable attacks.

## Gate T0 — Substrate integrity and no-source-dump discipline

**Plain-language view**

Before the paper-specific audit starts, the notebook has to show that its own helper layer is honest and that it is not sneaking the paper body into the notebook under the guise of setup.

**Claim being audited:** the notebook substrate is explicit, executable, and clean enough to support a serious CF18 audit.

**Why this is an honest attack**

If the helper layer is vague, missing, or mixed with source-dump behavior, every later result is less trustworthy. This gate attacks that failure mode first.

**Pass criteria**
- the ledger and terminal logger exist,
- the result record carries claim anchors and covered paper gates,
- helper checks all pass,
- a visible figure is emitted.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Any, Dict, List
from pathlib import Path
import hashlib
import json
import math

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.set_printoptions(precision=10, suppress=True)

@dataclass
class GateResult:
    gate_id: str
    gate_name: str
    canon_anchor: str
    covered_paper_gates: List[str]
    passed: bool
    metrics: Dict[str, Any]
    pass_criteria: Dict[str, Any]
    notes: str

LEDGER: List[GateResult] = []

def sha256_short(path: str) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

def terminal_log(
    gate_id: str,
    title: str,
    canon_anchor: str,
    covered_paper_gates: List[str],
    passed: bool,
    metrics: Dict[str, Any],
    criteria: Dict[str, Any],
    notes: str = "",
) -> None:
    status_symbol = "✅" if passed else "❌"
    print("=" * 100)
    print(f"[{gate_id}] {title}")
    print("- anchor:", canon_anchor)
    print("- covered paper gates:", ", ".join(covered_paper_gates) if covered_paper_gates else "(not paper-specific)")
    print("- status:", f"{status_symbol} {'PASS' if passed else 'FAIL'}")
    print("- criteria:")
    for k, v in criteria.items():
        print(f"    * {k}: {v}")
    print("- metrics:")
    for k, v in metrics.items():
        print(f"    * {k}: {v}")
    if notes:
        print("- notes:", notes)
    print("=" * 100)

helper_checks = {
    "GateResult_defined": int("GateResult" in globals()),
    "LEDGER_initialized_empty": int(isinstance(LEDGER, list) and len(LEDGER) == 0),
    "terminal_log_defined": int(callable(terminal_log)),
    "sha256_short_defined": int(callable(sha256_short)),
    "result_record_has_anchor": int("canon_anchor" in GateResult.__annotations__),
    "result_record_has_paper_gate_coverage": int("covered_paper_gates" in GateResult.__annotations__),
}

fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.bar(list(helper_checks.keys()), list(helper_checks.values()))
ax.set_ylim(0, 1.2)
ax.set_title("T0 substrate integrity checks")
ax.set_ylabel("PASS = 1")
ax.tick_params(axis="x", rotation=30)
ax.grid(alpha=0.25)
plt.show()

metrics = {k: bool(v) for k, v in helper_checks.items()}
criteria = {k: True for k in helper_checks}
passed = all(helper_checks.values())

terminal_log(
    "T0",
    "Substrate integrity and no-source-dump discipline",
    "Notebook contract",
    [],
    passed,
    metrics,
    criteria,
    notes="This gate checks the executable substrate only. The paper-specific no-source-dump burden is re-attacked at T1 on the manifest itself.",
)
LEDGER.append(
    GateResult(
        gate_id="T0",
        gate_name="Substrate integrity and no-source-dump discipline",
        canon_anchor="Notebook contract",
        covered_paper_gates=[],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Executable helper layer established.",
    )
)

## Paper-spec manifest

This manifest is the burden contract for CF18. It names the paper, source anchors, paper-level gates, and the exact claims this notebook is going to attack.

The manifest is intentionally lean. It contains the paper's burdens, not the paper's body.

In [ ]:
PAPER_SPEC = {
    "paper_id": "CF18",
    "paper_title": "Scale Re-Articulation, Effective Action, Hidden-Return Corrections, Running Couplings, Renormalization, and Corrected Symmetry Structure in VDM",
    "primitive_driver_id": "A(-1)",
    "primitive_driver_reference": {
        "path": "/mnt/data/A(-1).tex",
        "fingerprint_sha256_16": "2359c5150436fe3f",
    },
    "source_reference": {
        "paper_path": "/mnt/data/CF18.tex",
        "paper_fingerprint_sha256_16": "f50e0afa228d5950",
                        "paper_doi": "10.5281/zenodo.19154453",
    },
    "source_scope_note": "Audit targets the March 21, 2026 CF18 TeX source on disk. The notebook attacks only executable burdens and keeps the source external.",
    "paper_validation_gates": [
        {"gate_id": "G1", "description": "Reduced-description closure"},
        {"gate_id": "G2", "description": "Effective functional closure"},
        {"gate_id": "G3", "description": "Hidden-return correction closure"},
        {"gate_id": "G4", "description": "Running-coupling closure"},
        {"gate_id": "G5", "description": "Renormalization closure"},
        {"gate_id": "G6", "description": "Corrected symmetry / anomaly closure"},
        {"gate_id": "G7", "description": "Electroweak corrected-observable lift"},
        {"gate_id": "G8", "description": "Confinement corrected-observable lift"},
    ],
    "claims": [
        {
            "claim_id": "CF18-C1",
            "canon_anchor": "§3 Definition 2; §5 Lemma hidden persistence",
            "plain_language_claim": "Hidden burden does not vanish when a description is reduced.",
            "claim_type": "structural exactness",
            "statement": "A lawful reduced-description map must preserve hidden burden as visible induced consequence rather than erase it.",
            "attack_mode": "exact residual check on full-vs-reduced minimization; negative control by decoupling the hidden sector",
            "attack_honesty": "Uses a quadratic full model where reduction is exact. It tests the theorem's structural burden, not all nonlinear reductions.",
            "pass_criteria": {
                "reduction_residual_max": "<= 1e-12",
                "induced_shift_true_case": "> 1e-3",
                "induced_shift_decoupled_case": "<= 1e-12",
            },
            "fail_signature": "A nontrivial hidden coupling leaves no visible correction, or the reduced functional does not match the minimized full law.",
            "cannot_prove": "This does not prove every admissible nonlinear or nonperturbative reduction in the paper.",
            "figure_spec": "Residual curve plus true-vs-decoupled induced-shift comparison.",
            "result_metrics": ["reduction_residual_max", "induced_shift_true", "induced_shift_decoupled"],
            "expected_outputs": ["one residual figure", "one terminal PASS/FAIL block"],
            "paper_level_relevance": "G1",
            "covered_paper_gates": ["G1", "G2"],
        },
        {
            "claim_id": "CF18-C2",
            "canon_anchor": "§3 Hidden-return channel; §5 Lemma channels induce",
            "plain_language_claim": "Closed hidden-return channels add visible correction terms.",
            "claim_type": "constructive decomposition",
            "statement": "For an independent hidden-mode family, the induced correction equals the sum of channel contributions.",
            "attack_mode": "exact residual reconstruction plus channel-removal negative control",
            "attack_honesty": "Tests additivity in an explicit channelized model. It does not prove the most general interacting hidden sector.",
            "pass_criteria": {
                "channel_reconstruction_residual": "<= 1e-12",
                "all_channel_contributions_positive": True,
                "closed_channel_loss": "> 1e-4",
            },
            "fail_signature": "The direct correction cannot be reconstructed from the declared hidden-return channels.",
            "cannot_prove": "Does not settle nonlinear channel interference.",
            "figure_spec": "Bar chart of hidden-channel contributions.",
            "result_metrics": ["channel_reconstruction_residual", "closed_channel_loss"],
            "expected_outputs": ["one contribution figure", "one terminal PASS/FAIL block"],
            "paper_level_relevance": "G3",
            "covered_paper_gates": ["G3"],
        },
        {
            "claim_id": "CF18-C3",
            "canon_anchor": "§3 beta definition; §5 Running-coupling theorem",
            "plain_language_claim": "Visible coefficients must run when the explicit/hidden boundary moves, except at fixed points.",
            "claim_type": "scale-flow law",
            "statement": "A scale-indexed visible coefficient g(μ) changes with μ according to β(μ) = μ dg/dμ = dg/d ln μ, with β≈0 on plateaus where hidden content is unchanged.",
            "attack_mode": "smooth scale sweep, derivative consistency check, fixed-point/plateau probe",
            "attack_honesty": "Uses a smooth surrogate boundary to make β measurable. It attacks the running law, not full RG phenomenology.",
            "pass_criteria": {
                "beta_consistency_max_abs_error": "<= 2e-3",
                "left_plateau_beta_abs_max": "<= 5e-3",
                "right_plateau_beta_abs_max": "<= 5e-3",
                "g_monotone": True,
            },
            "fail_signature": "Moving the boundary changes hidden content without a corresponding coefficient flow, or β does not match the actual scale derivative.",
            "cannot_prove": "Does not derive sector-specific loop coefficients from first principles.",
            "figure_spec": "Running coefficient and beta-function curves over a log scale sweep.",
            "result_metrics": ["beta_consistency_max_abs_error", "left_plateau_beta_abs_max", "right_plateau_beta_abs_max", "g_monotone"],
            "expected_outputs": ["running curve", "beta curve", "terminal PASS/FAIL block"],
            "paper_level_relevance": "G4",
            "covered_paper_gates": ["G4"],
        },
        {
            "claim_id": "CF18-C4",
            "canon_anchor": "§3 renormalization condition / counterterm; §5 Renormalization theorem",
            "plain_language_claim": "Counterterms are forced when hidden-return corrections displace normalization conditions.",
            "claim_type": "normalization restoration",
            "statement": "If the corrected reduced observable misses its normalization target, a compensating counterterm can restore it exactly at the reference scale.",
            "attack_mode": "normalization residual before/after counterterm; wrong-counterterm negative control",
            "attack_honesty": "Tests the compensator logic on one explicit normalization condition. It does not prove all renormalization schemes.",
            "pass_criteria": {
                "normalization_residual_before": "> 1e-2",
                "normalization_residual_after": "<= 1e-12",
                "wrong_counterterm_residual": "> 1e-3",
            },
            "fail_signature": "The normalization target is restored without a compensator, or a compensator cannot restore it.",
            "cannot_prove": "Does not address all-orders renormalizability.",
            "figure_spec": "Observable flow with and without counterterm at the reference scale.",
            "result_metrics": ["normalization_residual_before", "normalization_residual_after", "wrong_counterterm_residual"],
            "expected_outputs": ["one normalization figure", "one terminal PASS/FAIL block"],
            "paper_level_relevance": "G5",
            "covered_paper_gates": ["G5"],
        },
        {
            "claim_id": "CF18-C5",
            "canon_anchor": "§5 Corrected symmetry and anomaly theorem",
            "plain_language_claim": "A corrected reduced layer falls into exactly one of three symmetry fates: preserved, removable drift, or anomaly.",
            "claim_type": "trichotomy classification",
            "statement": "Within a declared reduced family, preserved symmetry has zero removable shift, drift is exactly re-expressible inside the family, and anomaly leaves an irreducible residual.",
            "attack_mode": "three-case constructive classification with best-fit removable-family search",
            "attack_honesty": "The anomaly test is family-relative. It attacks the theorem's trichotomy logic, not every physical anomaly class.",
            "pass_criteria": {
                "preserved_fit_residual": "<= 1e-12",
                "preserved_shift_abs": "<= 1e-12",
                "drift_fit_residual": "<= 1e-12",
                "drift_shift_abs": ">= 1e-2",
                "anomaly_fit_residual": ">= 1e-3",
            },
            "fail_signature": "A genuinely non-removable obstruction can be absorbed into the allowed reduced family, or a removable shift cannot be absorbed.",
            "cannot_prove": "Does not compute anomaly coefficients for a real field theory.",
            "figure_spec": "Best-fit residuals for preserved, drift, and anomaly cases.",
            "result_metrics": ["preserved_fit_residual", "preserved_shift_abs", "drift_fit_residual", "drift_shift_abs", "anomaly_fit_residual"],
            "expected_outputs": ["one residual figure", "one terminal PASS/FAIL block"],
            "paper_level_relevance": "G6",
            "covered_paper_gates": ["G6"],
        },
        {
            "claim_id": "CF18-C6",
            "canon_anchor": "§5 Corollary sector-specific corrected observable",
            "plain_language_claim": "Earlier closed sectors can be lifted into scale-indexed corrected coefficient laws without reopening their structural closure.",
            "claim_type": "sector lift",
            "statement": "Electroweak-like mass data and confinement-like string-tension data can both be represented by the same reduced-description correction family, while a scale-independent model fails.",
            "attack_mode": "two-sector constructive fit plus constant-model negative control",
            "attack_honesty": "This is a corollary-level representability check, not a sector phenomenology notebook.",
            "pass_criteria": {
                "electroweak_lift_residual": "<= 1e-12",
                "confinement_lift_residual": "<= 1e-12",
                "electroweak_constant_model_error": ">= 1e-3",
                "confinement_constant_model_error": ">= 1e-3",
            },
            "fail_signature": "A corrected sector observable cannot be expressed as a scale-indexed visible coefficient law.",
            "cannot_prove": "Does not derive real electroweak or confinement running from the upstream CF stack here.",
            "figure_spec": "Two sector observables fitted by the same correction family.",
            "result_metrics": ["electroweak_lift_residual", "confinement_lift_residual", "electroweak_constant_model_error", "confinement_constant_model_error"],
            "expected_outputs": ["one sector-fit figure", "one terminal PASS/FAIL block"],
            "paper_level_relevance": "G7-G8",
            "covered_paper_gates": ["G7", "G8"],
        },
    ],
}

print(json.dumps(PAPER_SPEC["source_reference"], indent=2))
print(f"Claim count: {len(PAPER_SPEC['claims'])}")
print("Declared paper gates:", ", ".join(g["gate_id"] for g in PAPER_SPEC["paper_validation_gates"]))

## Gate T1 — Manifest completeness, burden coverage, and source-dump resistance

**Plain-language view**

Before any theorem attack runs, the notebook has to prove that the manifest is complete and that it is not secretly carrying the paper body.

**Claim being audited:** the CF18 manifest is explicit enough to support a real audit and lean enough to avoid source-dump behavior.

**Why this is an honest attack**

A notebook can look rigorous while quietly omitting burdens or cloning the paper into a giant manifest blob. This gate checks both failure modes directly.

**Pass criteria**
- required top-level fields exist,
- every claim carries the required burden fields,
- all paper validation gates G1–G8 are declared somewhere in claim coverage,
- the manifest stays lean and contains no raw TeX body markers,
- a visible figure is emitted.

In [ ]:
required_top_fields = {
    "paper_id",
    "paper_title",
    "primitive_driver_id",
    "primitive_driver_reference",
    "source_reference",
    "source_scope_note",
    "paper_validation_gates",
    "claims",
}
required_claim_fields = {
    "claim_id",
    "canon_anchor",
    "plain_language_claim",
    "claim_type",
    "statement",
    "attack_mode",
    "attack_honesty",
    "pass_criteria",
    "fail_signature",
    "cannot_prove",
    "figure_spec",
    "result_metrics",
    "expected_outputs",
    "paper_level_relevance",
    "covered_paper_gates",
}

top_field_ok = required_top_fields.issubset(PAPER_SPEC.keys())
missing_claim_fields = {
    claim["claim_id"]: sorted(list(required_claim_fields.difference(claim.keys())))
    for claim in PAPER_SPEC["claims"]
    if required_claim_fields.difference(claim.keys())
}

declared_gates = {g["gate_id"] for g in PAPER_SPEC["paper_validation_gates"]}
covered_gates = set()
for claim in PAPER_SPEC["claims"]:
    covered_gates.update(claim["covered_paper_gates"])

manifest_blob = json.dumps(PAPER_SPEC, indent=2)
raw_tex_markers = [
    "\\begin{document}",
    "\\section{",
    "\\subsection{",
    "\\bibliography{",
    "\\maketitle",
]
raw_tex_hits = sum(marker in manifest_blob for marker in raw_tex_markers)
total_manifest_chars = len(manifest_blob)
max_claim_statement_chars = max(len(claim["statement"]) for claim in PAPER_SPEC["claims"])

checks = {
    "top_fields_present": int(top_field_ok),
    "all_claim_fields_present": int(len(missing_claim_fields) == 0),
    "all_paper_gates_declared_and_covered": int(declared_gates == covered_gates),
    "raw_tex_body_markers_absent": int(raw_tex_hits == 0),
    "manifest_char_budget_under_15000": int(total_manifest_chars < 15000),
    "max_claim_statement_under_400": int(max_claim_statement_chars < 400),
}

fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.bar(list(checks.keys()), list(checks.values()))
ax.set_ylim(0, 1.2)
ax.set_title("T1 manifest completeness and source-dump resistance")
ax.set_ylabel("PASS = 1")
ax.tick_params(axis="x", rotation=30)
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "missing_top_fields": sorted(list(required_top_fields.difference(PAPER_SPEC.keys()))),
    "claims_missing_fields": missing_claim_fields,
    "declared_gates": sorted(list(declared_gates)),
    "covered_gates": sorted(list(covered_gates)),
    "raw_tex_hits": int(raw_tex_hits),
    "total_manifest_chars": int(total_manifest_chars),
    "max_claim_statement_chars": int(max_claim_statement_chars),
}
criteria = {
    "missing_top_fields": "[]",
    "claims_missing_fields": "{}",
    "declared_gates_equals_covered_gates": True,
    "raw_tex_hits": 0,
    "total_manifest_chars": "< 15000",
    "max_claim_statement_chars": "< 400",
}
passed = all(checks.values())

terminal_log(
    "T1",
    "Manifest completeness, burden coverage, and source-dump resistance",
    "Manifest contract",
    sorted(list(declared_gates)),
    passed,
    metrics,
    criteria,
    notes="This is the paper-specific anti-bloat gate. The manifest stays small and declarative.",
)
LEDGER.append(
    GateResult(
        gate_id="T1",
        gate_name="Manifest completeness, burden coverage, and source-dump resistance",
        canon_anchor="Manifest contract",
        covered_paper_gates=sorted(list(declared_gates)),
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Manifest completeness and anti-source-dump checks.",
    )
)

## Gate T2 — Audit-plan strength against soft-audit failure modes

**Plain-language view**

A complete manifest can still be weak. This gate checks whether the planned attacks actually have teeth.

**Claim being audited:** the CF18 audit plan is strong enough that it is not downgrading the paper into a toy explainer.

**Why this is an honest attack**

CF papers can fail by being over-polite. This gate insists on residual checks, negative controls, scale sweeps or boundary moves, exact thresholding, and explicit honesty limits.

**Pass criteria**
- the plan includes exact residual attacks,
- the plan includes negative controls,
- the plan includes a sweep / boundary-move style attack,
- the plan includes a removability / non-removability attack,
- every claim has thresholds and honesty limits,
- a visible figure is emitted.

In [ ]:
attack_modes = " | ".join(str(c["attack_mode"]).lower() for c in PAPER_SPEC["claims"])
attack_honesty_blob = " | ".join(str(c["attack_honesty"]).lower() for c in PAPER_SPEC["claims"])

attack_family_hits = {
    "exact_residual": int("residual" in attack_modes or "exact" in attack_modes),
    "negative_control": int("negative control" in attack_modes),
    "sweep_or_boundary_move": int("sweep" in attack_modes or "boundary" in attack_modes),
    "removability_attack": int("removable" in attack_modes or "non-removable" in attack_modes or "re-expressible" in attack_modes),
    "honesty_limits_explicit": int(all(bool(str(c["cannot_prove"]).strip()) for c in PAPER_SPEC["claims"])),
    "thresholds_present_for_all_claims": int(all(bool(c["pass_criteria"]) for c in PAPER_SPEC["claims"])),
}

vague_claims = [
    c["claim_id"] for c in PAPER_SPEC["claims"]
    if len(str(c["attack_mode"]).strip()) < 20 or len(str(c["attack_honesty"]).strip()) < 20
]
paper_gate_count = len({g for c in PAPER_SPEC["claims"] for g in c["covered_paper_gates"]})

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.bar(list(attack_family_hits.keys()), list(attack_family_hits.values()))
ax.set_ylim(0, 1.2)
ax.set_title("T2 attack-plan strength")
ax.set_ylabel("PASS = 1")
ax.tick_params(axis="x", rotation=30)
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "attack_family_hits": {k: bool(v) for k, v in attack_family_hits.items()},
    "vague_claims": vague_claims,
    "covered_paper_gate_count": int(paper_gate_count),
    "claim_count": int(len(PAPER_SPEC["claims"])),
}
criteria = {
    "all_attack_family_hits": True,
    "vague_claims": "[]",
    "covered_paper_gate_count": ">= 8",
}
passed = all(attack_family_hits.values()) and len(vague_claims) == 0 and paper_gate_count >= 8

terminal_log(
    "T2",
    "Audit-plan strength against soft-audit failure modes",
    "Manifest attack plan",
    sorted({g for c in PAPER_SPEC["claims"] for g in c["covered_paper_gates"]}),
    passed,
    metrics,
    criteria,
    notes="This gate checks the planned attack families, not the paper claims themselves.",
)
LEDGER.append(
    GateResult(
        gate_id="T2",
        gate_name="Audit-plan strength against soft-audit failure modes",
        canon_anchor="Manifest attack plan",
        covered_paper_gates=sorted({g for c in PAPER_SPEC["claims"] for g in c["covered_paper_gates"]}),
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Attack families and honesty discipline verified.",
    )
)

## Gate CF18-G1/G2 — Reduction closure and effective functional exactness

**Plain-language view**

The paper says reduction is not erasure. If the hidden sector still carries burden, the reduced visible law must change. This attack builds the simplest exact model where that statement can fail loudly if it is wrong.

**Claim being audited:** a lawful reduction with nonzero hidden coupling produces a nontrivial visible correction, and the reduced effective functional matches the minimized full law exactly.

**Why this is an honest attack**

This is not a verbal analogy. The full law is minimized over the hidden variables and compared directly against the proposed effective visible law. A decoupled negative control is included.

**Pass criteria**
- full-vs-reduced residual is below tolerance across the sampled visible domain,
- the coupled case induces a visible coefficient shift,
- the decoupled case induces no shift.

**Honesty limit**

This attacks the theorem on an exact quadratic reduction where the Schur-complement reduction is closed. It does not prove every nonlinear hidden sector in the paper.

In [ ]:
a = 3.0
b = np.array([0.8, -0.4, 0.5], dtype=float)
C = np.array([
    [2.5, 0.2, 0.0],
    [0.2, 1.8, 0.1],
    [0.0, 0.1, 1.6],
], dtype=float)
Cinv = np.linalg.inv(C)

xs = np.linspace(-2.0, 2.0, 401)
full_min = []
seff_vals = []

for x in xs:
    y_star = -Cinv @ b * x
    s_full = 0.5 * a * x**2 + x * (b @ y_star) + 0.5 * (y_star @ C @ y_star)
    s_eff = 0.5 * (a - b @ Cinv @ b) * x**2
    full_min.append(s_full)
    seff_vals.append(s_eff)

full_min = np.array(full_min)
seff_vals = np.array(seff_vals)
residual = full_min - seff_vals

b0 = np.zeros_like(b)
induced_shift_true = float(b @ Cinv @ b)
induced_shift_decoupled = float(b0 @ Cinv @ b0)
residual_max = float(np.max(np.abs(residual)))

fig, ax = plt.subplots(figsize=(6.8, 3.2))
ax.plot(xs, residual)
ax.set_title("CF18-G1/G2 full-vs-reduced residual")
ax.set_xlabel("visible variable x")
ax.set_ylabel("S_full,min - S_eff")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ax.bar(["coupled", "decoupled"], [induced_shift_true, induced_shift_decoupled])
ax.set_title("Induced visible coefficient shift")
ax.set_ylabel("shift magnitude")
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "reduction_residual_max": residual_max,
    "induced_shift_true": induced_shift_true,
    "induced_shift_decoupled": induced_shift_decoupled,
    "sample_count": int(len(xs)),
}
criteria = {
    "reduction_residual_max": "<= 1e-12",
    "induced_shift_true": "> 1e-3",
    "induced_shift_decoupled": "<= 1e-12",
    "sample_count": ">= 200",
}
passed = (
    residual_max <= 1e-12
    and induced_shift_true > 1e-3
    and induced_shift_decoupled <= 1e-12
    and len(xs) >= 200
)

terminal_log(
    "CF18-G1G2",
    "Reduction closure and effective functional exactness",
    "§3 Definition 2 / §5 hidden persistence + effective functional",
    ["G1", "G2"],
    passed,
    metrics,
    criteria,
    notes="The induced correction is the Schur-complement shift of the visible quadratic coefficient.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G1G2",
        gate_name="Reduction closure and effective functional exactness",
        canon_anchor="§3 Definition 2 / §5 hidden persistence + effective functional",
        covered_paper_gates=["G1", "G2"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Exact quadratic reduction test with decoupled negative control.",
    )
)

## Gate CF18-G3 — Hidden-return correction additivity

**Plain-language view**

The paper does not just say hidden structure matters. It says closed hidden-return channels contribute visible correction terms. This attack checks whether the correction really decomposes into channel pieces.

**Claim being audited:** the induced correction can be reconstructed as the sum of admissible hidden-return channel contributions in a channelized hidden sector.

**Why this is an honest attack**

The channel sum is compared against the direct correction from the full hidden block. One channel is then closed as a negative control to force a visible loss.

**Pass criteria**
- direct and channelized corrections agree to tolerance,
- all declared channel contributions are positive in this model,
- closing one channel changes the visible correction by a visible amount.

**Honesty limit**

This tests the additive channel law on an independent hidden-mode family. It does not resolve nonlinear hidden-channel interference.

In [ ]:
masses = np.array([1.4, 2.1, 3.7, 5.3], dtype=float)
couplings = np.array([0.50, 0.35, 0.20, 0.10], dtype=float)
C = np.diag(masses**2)
Cinv = np.linalg.inv(C)

direct_correction = float(couplings @ Cinv @ couplings)
channel_contributions = (couplings**2) / (masses**2)
reconstructed_correction = float(np.sum(channel_contributions))
reconstruction_residual = float(abs(direct_correction - reconstructed_correction))

closed_index = 1
couplings_closed = couplings.copy()
couplings_closed[closed_index] = 0.0
direct_closed = float(couplings_closed @ Cinv @ couplings_closed)
closed_channel_loss = float(direct_correction - direct_closed)

fig, ax = plt.subplots(figsize=(6.6, 3.2))
labels = [f"C{i+1}" for i in range(len(channel_contributions))]
ax.bar(labels, channel_contributions)
ax.axhline(direct_correction, linestyle="--", linewidth=1.0, label="direct total")
ax.set_title("CF18-G3 hidden-return channel contributions")
ax.set_ylabel("contribution to visible correction")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "channel_reconstruction_residual": reconstruction_residual,
    "all_channel_contributions_positive": bool(np.all(channel_contributions > 0)),
    "closed_channel_loss": closed_channel_loss,
    "direct_correction": direct_correction,
}
criteria = {
    "channel_reconstruction_residual": "<= 1e-12",
    "all_channel_contributions_positive": True,
    "closed_channel_loss": "> 1e-4",
}
passed = (
    reconstruction_residual <= 1e-12
    and np.all(channel_contributions > 0)
    and closed_channel_loss > 1e-4
)

terminal_log(
    "CF18-G3",
    "Hidden-return correction additivity",
    "§3 hidden-return channel / §5 channels induce",
    ["G3"],
    passed,
    metrics,
    criteria,
    notes="In this diagonal hidden sector, the direct correction is exactly the sum of channel contributions.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G3",
        gate_name="Hidden-return correction additivity",
        canon_anchor="§3 hidden-return channel / §5 channels induce",
        covered_paper_gates=["G3"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Channel-sum reconstruction with one-channel closure negative control.",
    )
)

## Gate CF18-G4 — Running-coupling closure under a moving explicit/hidden boundary

**Plain-language view**

If the scale boundary moves, the visible coefficient should flow. If it does not, the paper's running law is empty. This attack measures that flow directly.

**Claim being audited:** a scale-indexed visible coefficient obeys the beta-function law, and beta collapses toward zero where the hidden content stops changing.

**Why this is an honest attack**

The notebook uses a smooth hidden-weight boundary so the running coefficient and its beta function can be measured over a dense sweep. The analytic and numerical beta curves are compared directly.

**Pass criteria**
- beta from the derivative matches beta from the hidden-weight formula,
- beta is small on both plateaus,
- the visible coefficient is monotone under the chosen scale move.

**Honesty limit**

This is a structural running-law attack, not a full renormalization-group derivation for a physical sector.

In [ ]:
modes = np.array([1.0, 2.0, 4.0, 8.0, 16.0], dtype=float)
weights = np.array([0.30, 0.18, 0.10, 0.06, 0.04], dtype=float)
sigma = 0.18
mu = np.logspace(-1, 2, 500)
ln_mu = np.log(mu)
ln_modes = np.log(modes)

hidden_weight = 1.0 / (1.0 + np.exp(-(ln_modes[None, :] - ln_mu[:, None]) / sigma))
g0 = 1.0
g_eff = g0 + hidden_weight @ weights

beta_analytic = np.sum(weights[None, :] * (-(hidden_weight * (1.0 - hidden_weight)) / sigma), axis=1)
beta_numeric = np.gradient(g_eff, ln_mu)

beta_consistency_max_abs_error = float(np.max(np.abs(beta_numeric - beta_analytic)))
left_plateau_beta_abs_max = float(np.max(np.abs(beta_numeric[mu < 0.15])))
right_plateau_beta_abs_max = float(np.max(np.abs(beta_numeric[mu > 70.0])))
g_monotone = bool(np.all(np.diff(g_eff) <= 1e-10))

fig, ax = plt.subplots(figsize=(6.8, 3.2))
ax.semilogx(mu, g_eff)
ax.set_title("CF18-G4 running visible coefficient")
ax.set_xlabel("scale μ")
ax.set_ylabel("g_eff(μ)")
ax.grid(alpha=0.25, which="both")
plt.show()

fig, ax = plt.subplots(figsize=(6.8, 3.2))
ax.semilogx(mu, beta_numeric, label="numeric β")
ax.semilogx(mu, beta_analytic, linestyle="--", label="analytic β")
ax.set_title("CF18-G4 beta-function consistency")
ax.set_xlabel("scale μ")
ax.set_ylabel("β(μ) = d g / d ln μ")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()

metrics = {
    "beta_consistency_max_abs_error": beta_consistency_max_abs_error,
    "left_plateau_beta_abs_max": left_plateau_beta_abs_max,
    "right_plateau_beta_abs_max": right_plateau_beta_abs_max,
    "g_monotone": g_monotone,
}
criteria = {
    "beta_consistency_max_abs_error": "<= 2e-3",
    "left_plateau_beta_abs_max": "<= 5e-3",
    "right_plateau_beta_abs_max": "<= 5e-3",
    "g_monotone": True,
}
passed = (
    beta_consistency_max_abs_error <= 2e-3
    and left_plateau_beta_abs_max <= 5e-3
    and right_plateau_beta_abs_max <= 5e-3
    and g_monotone
)

terminal_log(
    "CF18-G4",
    "Running-coupling closure under a moving explicit/hidden boundary",
    "§3 beta definition / §5 running-coupling theorem",
    ["G4"],
    passed,
    metrics,
    criteria,
    notes="β is computed both from the hidden-weight law and from the numerical derivative of g_eff.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G4",
        gate_name="Running-coupling closure under a moving explicit/hidden boundary",
        canon_anchor="§3 beta definition / §5 running-coupling theorem",
        covered_paper_gates=["G4"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Dense log-scale sweep with plateau probes.",
    )
)

## Gate CF18-G5 — Renormalization-condition restoration by forced compensator

**Plain-language view**

The paper says counterterms are not optional bookkeeping. They are forced when the reduced observable misses its normalization target. This attack checks that logic directly.

**Claim being audited:** if hidden-return corrections shift a target observable away from its normalization condition, a compensating counterterm restores it exactly at the reference scale.

**Why this is an honest attack**

The notebook measures the normalization residual before compensation, after the exact compensator, and after an intentionally wrong compensator.

**Pass criteria**
- the uncorrected observable misses the target,
- the correct compensator restores the target to tolerance,
- the wrong compensator does not.

**Honesty limit**

This attacks one explicit normalization condition on one running observable. It does not settle all schemes or all-orders renormalizability.

In [ ]:
modes = np.array([1.0, 2.0, 4.0, 8.0, 16.0], dtype=float)
weights = np.array([0.30, 0.18, 0.10, 0.06, 0.04], dtype=float)
sigma = 0.18
mu = np.logspace(-1, 2, 500)
ln_mu = np.log(mu)
ln_modes = np.log(modes)
hidden_weight = 1.0 / (1.0 + np.exp(-(ln_modes[None, :] - ln_mu[:, None]) / sigma))
g0 = 1.0
g_eff = g0 + hidden_weight @ weights

mu0 = 3.0
g_phys = 1.185
g_eff_mu0 = float(np.interp(np.log(mu0), ln_mu, g_eff))
delta_ct = g_phys - g_eff_mu0
g_ren = g_eff + delta_ct

wrong_delta_ct = 0.5 * delta_ct
g_wrong = g_eff + wrong_delta_ct

normalization_residual_before = float(abs(g_eff_mu0 - g_phys))
normalization_residual_after = float(abs(np.interp(np.log(mu0), ln_mu, g_ren) - g_phys))
wrong_counterterm_residual = float(abs(np.interp(np.log(mu0), ln_mu, g_wrong) - g_phys))

fig, ax = plt.subplots(figsize=(6.9, 3.3))
ax.semilogx(mu, g_eff, label="uncorrected")
ax.semilogx(mu, g_ren, label="renormalized")
ax.semilogx(mu, g_wrong, linestyle="--", label="wrong counterterm")
ax.axhline(g_phys, color="k", linewidth=1.0, linestyle=":", label="normalization target")
ax.axvline(mu0, color="gray", linewidth=1.0, linestyle=":")
ax.set_title("CF18-G5 normalization restoration at μ0")
ax.set_xlabel("scale μ")
ax.set_ylabel("observable / coefficient")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()

metrics = {
    "normalization_residual_before": normalization_residual_before,
    "normalization_residual_after": normalization_residual_after,
    "wrong_counterterm_residual": wrong_counterterm_residual,
    "mu0": mu0,
}
criteria = {
    "normalization_residual_before": "> 1e-2",
    "normalization_residual_after": "<= 1e-12",
    "wrong_counterterm_residual": "> 1e-3",
}
passed = (
    normalization_residual_before > 1e-2
    and normalization_residual_after <= 1e-12
    and wrong_counterterm_residual > 1e-3
)

terminal_log(
    "CF18-G5",
    "Renormalization-condition restoration by forced compensator",
    "§3 renormalization condition / counterterm / §5 renormalization theorem",
    ["G5"],
    passed,
    metrics,
    criteria,
    notes="The counterterm is fixed by one declared normalization condition at the reference scale μ0.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G5",
        gate_name="Renormalization-condition restoration by forced compensator",
        canon_anchor="§3 renormalization condition / counterterm / §5 renormalization theorem",
        covered_paper_gates=["G5"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Exact-target restoration versus wrong-counterterm negative control.",
    )
)

## Gate CF18-G6 — Preserved symmetry, removable drift, and anomaly separation

**Plain-language view**

The paper says the corrected reduced layer has exactly three symmetry fates. This attack forces the notebook to separate those cases cleanly instead of talking around them.

**Claim being audited:** within a declared reduced family, the corrected layer distinguishes exact preservation, removable drift, and genuine anomaly.

**Why this is an honest attack**

The notebook gives the reduced layer an allowed family of re-expressions — shifted quadratic wells — and then asks which corrected potentials can be absorbed into that family. The cases are designed so one should stay exact, one should be removable by a finite family re-expression, and one should not.

**Pass criteria**
- the preserved case fits the family with zero shift,
- the drift case fits the family but with a nonzero shift,
- the anomaly case leaves an irreducible fit residual.

**Honesty limit**

This is a family-relative anomaly test. It attacks the theorem's trichotomy logic, not every anomaly in quantum field theory.

In [ ]:
x = np.linspace(-2.0, 2.0, 801)

y_preserved = 0.5 * x**2
y_drift = 0.5 * (x - 0.6)**2
y_anomaly = 0.5 * x**2 + 0.08 * x**3

def fit_shifted_quadratic(x, y):
    # Fit y ≈ a2 x^2 + a1 x + a0, which corresponds to 0.5*A*(x-p)^2 + c.
    coeff = np.polyfit(x, y, deg=2)
    a2, a1, a0 = coeff
    y_fit = np.polyval(coeff, x)
    resid = y - y_fit
    rms = float(np.sqrt(np.mean(resid**2)))
    if abs(a2) < 1e-14:
        shift = float("nan")
    else:
        A = 2.0 * a2
        shift = float(-a1 / (2.0 * a2))
    return coeff, y_fit, resid, rms, shift

coeff_p, fit_p, resid_p, rms_p, shift_p = fit_shifted_quadratic(x, y_preserved)
coeff_d, fit_d, resid_d, rms_d, shift_d = fit_shifted_quadratic(x, y_drift)
coeff_a, fit_a, resid_a, rms_a, shift_a = fit_shifted_quadratic(x, y_anomaly)

fig, ax = plt.subplots(figsize=(7.0, 3.3))
ax.plot(x, resid_p, label="preserved residual")
ax.plot(x, resid_d, label="drift residual")
ax.plot(x, resid_a, label="anomaly residual")
ax.set_title("CF18-G6 removable-family residuals")
ax.set_xlabel("x")
ax.set_ylabel("best-fit residual")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "preserved_fit_residual": rms_p,
    "preserved_shift_abs": float(abs(shift_p)),
    "drift_fit_residual": rms_d,
    "drift_shift_abs": float(abs(shift_d)),
    "anomaly_fit_residual": rms_a,
}
criteria = {
    "preserved_fit_residual": "<= 1e-12",
    "preserved_shift_abs": "<= 1e-12",
    "drift_fit_residual": "<= 1e-12",
    "drift_shift_abs": ">= 1e-2",
    "anomaly_fit_residual": ">= 1e-3",
}
passed = (
    rms_p <= 1e-12
    and abs(shift_p) <= 1e-12
    and rms_d <= 1e-12
    and abs(shift_d) >= 1e-2
    and rms_a >= 1e-3
)

terminal_log(
    "CF18-G6",
    "Preserved symmetry, removable drift, and anomaly separation",
    "§5 corrected symmetry and anomaly theorem",
    ["G6"],
    passed,
    metrics,
    criteria,
    notes="The allowed reduced family is the shifted-quadratic class. The anomaly case is defined relative to that admissible class.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G6",
        gate_name="Preserved symmetry, removable drift, and anomaly separation",
        canon_anchor="§5 corrected symmetry and anomaly theorem",
        covered_paper_gates=["G6"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Three-case constructive trichotomy inside a declared reduced family.",
    )
)

## Gate CF18-G7/G8 — Sector-lift representability for electroweak-like and confinement-like observables

**Plain-language view**

The paper's corollary says earlier closed sectors can be lifted into scale-indexed corrected coefficient laws. This attack checks that representability burden instead of just repeating the corollary.

**Claim being audited:** two different sector observables can both be expressed by the same reduced-description correction family, while a scale-independent model fails.

**Why this is an honest attack**

Both observables are fitted against the same hidden-weight basis used earlier for scale re-articulation. A constant-only negative control is included so the lift has to earn its keep.

**Pass criteria**
- the electroweak-like observable fits the correction family,
- the confinement-like observable fits the correction family,
- constant-only fits are visibly worse for both.

**Honesty limit**

This is a corollary-level representability test. It does not derive real electroweak or confinement running from the full upstream canon.

In [ ]:
modes = np.array([1.0, 2.0, 4.0, 8.0, 16.0], dtype=float)
sigma = 0.18
mu = np.logspace(-1, 2, 400)
ln_mu = np.log(mu)
ln_modes = np.log(modes)
hidden_weight = 1.0 / (1.0 + np.exp(-(ln_modes[None, :] - ln_mu[:, None]) / sigma))

basis = np.column_stack([np.ones_like(mu), hidden_weight])

theta_ew_true = np.array([1.40, 0.12, -0.05, 0.07, 0.00, 0.03])
theta_conf_true = np.array([0.65, 0.04, 0.07, 0.10, 0.02, 0.01])

obs_ew = basis @ theta_ew_true
obs_conf = basis @ theta_conf_true

theta_ew_fit, *_ = np.linalg.lstsq(basis, obs_ew, rcond=None)
theta_conf_fit, *_ = np.linalg.lstsq(basis, obs_conf, rcond=None)

obs_ew_fit = basis @ theta_ew_fit
obs_conf_fit = basis @ theta_conf_fit

ew_lift_residual = float(np.max(np.abs(obs_ew - obs_ew_fit)))
conf_lift_residual = float(np.max(np.abs(obs_conf - obs_conf_fit)))

const_basis = np.ones((len(mu), 1))
theta_ew_const, *_ = np.linalg.lstsq(const_basis, obs_ew, rcond=None)
theta_conf_const, *_ = np.linalg.lstsq(const_basis, obs_conf, rcond=None)
obs_ew_const = const_basis @ theta_ew_const
obs_conf_const = const_basis @ theta_conf_const

ew_constant_model_error = float(np.sqrt(np.mean((obs_ew - obs_ew_const)**2)))
conf_constant_model_error = float(np.sqrt(np.mean((obs_conf - obs_conf_const)**2)))

fig, ax = plt.subplots(figsize=(7.0, 3.3))
ax.semilogx(mu, obs_ew, label="electroweak-like target")
ax.semilogx(mu, obs_ew_fit, linestyle="--", label="electroweak-like fit")
ax.semilogx(mu, obs_conf, label="confinement-like target")
ax.semilogx(mu, obs_conf_fit, linestyle="--", label="confinement-like fit")
ax.set_title("CF18-G7/G8 sector-lift fits")
ax.set_xlabel("scale μ")
ax.set_ylabel("corrected observable")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()

metrics = {
    "electroweak_lift_residual": ew_lift_residual,
    "confinement_lift_residual": conf_lift_residual,
    "electroweak_constant_model_error": ew_constant_model_error,
    "confinement_constant_model_error": conf_constant_model_error,
}
criteria = {
    "electroweak_lift_residual": "<= 1e-12",
    "confinement_lift_residual": "<= 1e-12",
    "electroweak_constant_model_error": ">= 1e-3",
    "confinement_constant_model_error": ">= 1e-3",
}
passed = (
    ew_lift_residual <= 1e-12
    and conf_lift_residual <= 1e-12
    and ew_constant_model_error >= 1e-3
    and conf_constant_model_error >= 1e-3
)

terminal_log(
    "CF18-G7G8",
    "Sector-lift representability for electroweak-like and confinement-like observables",
    "§5 corollary sector-specific corrected observable",
    ["G7", "G8"],
    passed,
    metrics,
    criteria,
    notes="Both observables fit the same scale-indexed hidden-weight correction family, while constant-only fits fail visibly.",
)
LEDGER.append(
    GateResult(
        gate_id="CF18-G7G8",
        gate_name="Sector-lift representability for electroweak-like and confinement-like observables",
        canon_anchor="§5 corollary sector-specific corrected observable",
        covered_paper_gates=["G7", "G8"],
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Corollary-level representability attack with constant-model negative control.",
    )
)

## Gate T6 — Final results ledger and coverage closure

**Plain-language view**

At the end, the reader should not have to infer what happened. This cell closes the notebook with a visible ledger, visible coverage, and visible pass/fail counts.

**Claim being audited:** the notebook ends with an explicit paper-wide ledger rather than leaving coverage ambiguous.

**Why this is an honest attack**

Even a good set of cells can fail as an artifact if the ending blurs what passed, what failed, or what was never attacked. This gate makes that impossible to hide.

**Pass criteria**
- every prior gate appears exactly once in the ledger,
- pass/fail counts are consistent,
- paper validation gates G1–G8 are all covered somewhere in the ledger,
- a visible figure is emitted.

In [ ]:
ledger_df = pd.DataFrame([asdict(g) for g in LEDGER])

gate_ids = ledger_df["gate_id"].tolist()
pass_count = int(ledger_df["passed"].sum())
fail_count = int((~ledger_df["passed"]).sum())
covered_paper_gates = sorted({g for row in ledger_df["covered_paper_gates"] for g in row})
declared_paper_gates = sorted([g["gate_id"] for g in PAPER_SPEC["paper_validation_gates"]])
uncovered_paper_gates = sorted(list(set(declared_paper_gates).difference(set(covered_paper_gates))))

display_cols = ["gate_id", "canon_anchor", "covered_paper_gates", "passed"]
print(ledger_df[display_cols].to_string(index=False))

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.bar(gate_ids, [1 if p else 0 for p in ledger_df["passed"]])
ax.set_ylim(0, 1.2)
ax.set_title("Final CF18 audit ledger")
ax.set_ylabel("PASS = 1, FAIL = 0")
ax.tick_params(axis="x", rotation=30)
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "ledger_gate_count": int(len(ledger_df)),
    "unique_gate_count": int(len(set(gate_ids))),
    "pass_count": pass_count,
    "fail_count": fail_count,
    "declared_paper_gates": declared_paper_gates,
    "covered_paper_gates": covered_paper_gates,
    "uncovered_paper_gates": uncovered_paper_gates,
}
criteria = {
    "ledger_gate_count": ">= 1",
    "unique_gate_count_equals_ledger_gate_count": True,
    "pass_count_plus_fail_count_equals_ledger_gate_count": True,
    "uncovered_paper_gates": "[]",
}
passed = (
    len(ledger_df) >= 1
    and len(set(gate_ids)) == len(ledger_df)
    and pass_count + fail_count == len(ledger_df)
    and len(uncovered_paper_gates) == 0
)

terminal_log(
    "T6",
    "Final results ledger and coverage closure",
    "Notebook closure",
    declared_paper_gates,
    passed,
    metrics,
    criteria,
    notes="This gate checks artifact closure and paper-gate coverage, not any one theorem.",
)
LEDGER.append(
    GateResult(
        gate_id="T6",
        gate_name="Final results ledger and coverage closure",
        canon_anchor="Notebook closure",
        covered_paper_gates=declared_paper_gates,
        passed=passed,
        metrics=metrics,
        pass_criteria=criteria,
        notes="Artifact closure and coverage summary.",
    )
)